# BaseCallbackHandler

`BaseCallbackHandler` is the main base class used to create custom callback handlers in LangChain.
it combines callback methods from all six mixin classes. You only override the methods required by your application. The remaining methods are automatically inherited.

## Base Classes

* `LLMManagerMixin` — Handles LLM and chat-model events.
* `ChainManagerMixin` — Handles chain and agent events.
* `ToolManagerMixin` — Handles tool execution events.
* `RetrieverManagerMixin` — Handles retriever events.
* `CallbackManagerMixin` — Handles the starting events of models, chains, tools and retrievers.
* `RunManagerMixin` — Handles common runtime events such as retries, text and custom events.

## Attributes

* `raise_error` — Raises the callback error instead of silently ignoring it.
* `run_inline` — Runs the callback immediately in the same execution flow.
* `ignore_llm` — Ignores callbacks related to LLM execution.
* `ignore_retry` — Ignores callbacks related to retry attempts.
* `ignore_chain` — Ignores callbacks related to chain execution.
* `ignore_agent` — Ignores callbacks related to agent execution.
* `ignore_retriever` — Ignores callbacks related to document retrieval.
* `ignore_chat_model` — Ignores callbacks related specifically to chat models.
* `ignore_custom_event` — Ignores user-defined custom callback events.

> There is no common attribute named `ignore`. LangChain provides separate `ignore_*` attributes for different callback categories.

## Important Inherited Methods

### LLMManagerMixin Methods

* `on_llm_new_token()` — Runs whenever the model generates a new token during streaming.
* `on_llm_end()` — Runs after the LLM successfully finishes generating its response.
* `on_llm_error()` — Runs when an error occurs during LLM execution.
* `on_stream_event()` — Runs whenever a streaming protocol event is received.

### ChainManagerMixin Methods

* `on_chain_end()` — Runs after a chain completes successfully.
* `on_chain_error()` — Runs when an error occurs inside a chain.
* `on_agent_action()` — Runs when an agent decides to perform an action.
* `on_agent_finish()` — Runs when an agent completes its task.

### ToolManagerMixin Methods

* `on_tool_end()` — Runs after a tool completes successfully.
* `on_tool_error()` — Runs when an error occurs while executing a tool.

### RetrieverManagerMixin Methods

* `on_retriever_end()` — Runs after the retriever returns documents.
* `on_retriever_error()` — Runs when document retrieval fails.

### CallbackManagerMixin Methods

* `on_llm_start()` — Runs before a normal text-completion LLM starts.
* `on_chat_model_start()` — Runs before a chat model starts.
* `on_chain_start()` — Runs before a chain starts.
* `on_tool_start()` — Runs before a tool starts executing.
* `on_retriever_start()` — Runs before document retrieval begins.

### RunManagerMixin Methods

* `on_text()` — Runs when some intermediate text is produced.
* `on_retry()` — Runs whenever an operation is retried.
* `on_custom_event()` — Runs when a user-defined custom event is triggered.





In [2]:
from langchain_core.callbacks import BaseCallbackHandler  # Base handler
from langchain_core.language_models import FakeListLLM, FakeListChatModel  # Fake AI models
from langchain_core.runnables import RunnableLambda  # Simple chain

class MyHandler(BaseCallbackHandler):  # Custom handler
    raise_error = False  # Callback errors will not stop the AI
    run_inline = True  # Callback runs immediately

    @property
    def ignore_llm(self):  # Controls normal LLM callbacks
        return False  # Allow LLM callbacks

    @property
    def ignore_chat_model(self):  # Controls chat-model callbacks
        return True  # Ignore chat-model callbacks

    @property
    def ignore_chain(self):  # Controls chain callbacks
        return True  # Ignore chain callbacks

    def on_llm_start(self, serialized, prompts, **kwargs):  # LLM event
        print("LLM callback executed")

    def on_chat_model_start(self, serialized, messages, **kwargs):  # Chat event
        print("Chat callback executed")

    def on_chain_start(self, serialized, inputs, **kwargs):  # Chain event
        print("Chain callback executed")

handler = MyHandler()  # Creates handler

llm = FakeListLLM(responses=["Hello from LLM"])  # Fake normal LLM
chat = FakeListChatModel(responses=["Hello from chat"])  # Fake chat model
chain = RunnableLambda(lambda text: text.upper())  # Simple chain

print(llm.invoke("Hello", config={"callbacks": [handler]}))  # LLM callback runs
chat.invoke("Hello", config={"callbacks": [handler]})  # Chat callback is ignored
chain.invoke("Hello", config={"callbacks": [handler]})  # Chain callback is ignored

LLM callback executed
Hello from LLM


'HELLO'

## Inherited from RetriveManagerMixin
A retriever searches a data source and returns relevant documents. This mixin helps in monitoring, logging, debugging, and error handling during retrieval.
### Methods
1. `on_retriever_error`: Runs when the retriever encounters an error.
   * **Syntax:**
     ```python
     on_retriever_error(
         self,
         error: BaseException, # Error that occurred
         *,
         run_id: UUID, # ID of the current run
         parent_run_id: UUID | None = None, # ID of the parent run
         **kwargs: Any # Additional arguments
     ) -> Any
     ```

2. `on_retriever_end`: Runs when the retriever finishes successfully.
   * **Syntax:**
     ```python
     on_retriever_end(
         self,
         documents: Sequence[Document], # Documents retrieved
         *,
         run_id: UUID, # ID of the current run
         parent_run_id: UUID | None = None, # ID of the parent run
         **kwargs: Any # Additional arguments
     ) -> Any
     ```

In [1]:
from langchain_core.callbacks import BaseCallbackHandler  # Callback base class
from langchain_core.documents import Document  # Represents retrieved data
from langchain_core.retrievers import BaseRetriever  # Retriever base class

class RetrievalMonitor(BaseCallbackHandler):  # Monitors AI document retrieval
    def on_retriever_end(self, documents, **kwargs):  # Runs after successful search
        print("AI found:", len(documents), "documents")  # Shows number of results

    def on_retriever_error(self, error, **kwargs):  # Runs when search fails
        print("AI retrieval failed:", error)  # Shows retrieval error

class CompanyRetriever(BaseRetriever):  # Searches company information
    def _get_relevant_documents(self, query):  # Receives user's question
        if not query:  # Checks for an empty question
            raise ValueError("Question cannot be empty")  # Creates an error

        documents = [  # Knowledge available to the AI
            Document(page_content="Employees get 20 paid leaves."),
            Document(page_content="Work timings are 9 AM to 6 PM.")
        ]

        return [doc for doc in documents if "leave" in query.lower() and "leave" in doc.page_content.lower()]  # Returns matching data

monitor = RetrievalMonitor()  # Creates callback monitor
retriever = CompanyRetriever()  # Creates AI retriever

docs = retriever.invoke(  # AI searches for relevant information
    "How many leaves do employees get?",
    config={"callbacks": [monitor]}  # Connects the callback
)

print("Context for AI:", docs[0].page_content)  # Data later sent to the LLM

AI found: 1 documents
Context for AI: Employees get 20 paid leaves.


## Inherited from `LLMManagerMixin`
Mixin used to handle callbacks for LLM and chat-model events.
It is useful for monitoring streaming tokens, successful completion, errors, and stream events.
### Methods
1. `on_llm_new_token`: Runs whenever a new output token or content block is generated.
   * Works only when streaming is enabled.
   * Supports chat models and legacy text-completion models.
   - **Syntax:**
     ```python
     on_llm_new_token(
         self,
         token: str | list[str | dict[str, Any]], # New token or content blocks
         *,
         chunk: GenerationChunk | ChatGenerationChunk | None = None, # Generated chunk
         run_id: UUID, # ID of the current run
         parent_run_id: UUID | None = None, # ID of the parent run
         tags: list[str] | None = None, # Tags associated with the run
         **kwargs: Any # Additional arguments
     ) -> Any
     ```

2. `on_llm_end`: Runs when the LLM finishes successfully.
   * **Syntax:**
     ```python
     on_llm_end(
         self,
         response: LLMResult, # Generated LLM response
         *,
         run_id: UUID, # ID of the current run
         parent_run_id: UUID | None = None, # ID of the parent run
         tags: list[str] | None = None, # Tags associated with the run
         **kwargs: Any # Additional arguments
     ) -> Any
     ```

3. `on_llm_error`: Runs when the LLM encounters an error.
   * **Syntax:**
     ```python
     on_llm_error(
         self,
         error: BaseException, # Error that occurred
         *,
         run_id: UUID, # ID of the current run
         parent_run_id: UUID | None = None, # ID of the parent run
         tags: list[str] | None = None, # Tags associated with the run
         **kwargs: Any # Additional arguments
     ) -> Any
     ```

4. `on_stream_event`: Runs for every protocol event generated by `stream_events(version="v3")` or `astream_events(version="v3")`.
   * Fires for message start, content-block start, content-block updates, content-block finish, and message finish.
   * It does not run with `stream()` or `astream()`; use `on_llm_new_token` for those.
   - **Syntax:**
     ```python
     on_stream_event(
         self,
         event: MessagesData, # Streaming protocol event
         *,
         run_id: UUID, # ID of the current run
         parent_run_id: UUID | None = None, # ID of the parent run
         tags: list[str] | None = None, # Tags associated with the run
         **kwargs: Any # Additional arguments
     ) -> Any
     ```

It is triggered for message-start and per-block content events.



In [ ]:
from langchain_openai import ChatOpenAI  # AI model
from langchain_core.callbacks import BaseCallbackHandler  # Callback base class

class MyHandler(BaseCallbackHandler):  # Our callback handler
    def on_llm_new_token(self, token, **kwargs):  # Runs for each new token
        print(token, end="")

    def on_llm_end(self, response, **kwargs):  # Runs when AI finishes
        print("\nAI finished")

    def on_llm_error(self, error, **kwargs):  # Runs when AI fails
        print("\nError:", error)

    def on_stream_event(self, event, **kwargs):  # Runs for each v3 event
        print("Stream event received")

handler = MyHandler()  # Creates handler
model = ChatOpenAI(model="gpt-4.1-mini", streaming=True)  # Creates model

for chunk in model.stream("Say hello", config={"callbacks": [handler]}):  # Token streaming
    pass

for event in model.stream_events("Say hello", version="v3", config={"callbacks": [handler]}):  # Event streaming
    pass

try:
    bad_model = ChatOpenAI(api_key="wrong-key", max_retries=0)  # Invalid model
    bad_model.invoke("Hello", config={"callbacks": [handler]})  # Creates error
except Exception:
    pass

## Inherited from `ChainManagerMixin`

Mixin used to handle callbacks for chain and agent events.
It is useful for monitoring chain completion, chain errors, agent actions, and agent completion.

### Methods

1. `on_chain_end`: Runs when the chain finishes successfully.

   * Receives the final output produced by the chain.

   - **Syntax:**

     ```python
     on_chain_end(
         self,
         outputs: dict[str, Any], # Final output of the chain
         *,
         run_id: UUID, # ID of the current chain run
         parent_run_id: UUID | None = None, # ID of the parent run
         **kwargs: Any # Additional arguments
     ) -> Any
     ```

2. `on_chain_error`: Runs when an error occurs during chain execution.

   * Helps in logging and debugging chain failures.

   - **Syntax:**

     ```python
     on_chain_error(
         self,
         error: BaseException, # Error that occurred
         *,
         run_id: UUID, # ID of the current chain run
         parent_run_id: UUID | None = None, # ID of the parent run
         **kwargs: Any # Additional arguments
     ) -> Any
     ```

3. `on_agent_action`: Runs whenever an agent decides to perform an action.

   * An action usually means calling a tool or performing a specific task.

   - **Syntax:**

     ```python
     on_agent_action(
         self,
         action: AgentAction, # Action selected by the agent
         *,
         run_id: UUID, # ID of the current agent run
         parent_run_id: UUID | None = None, # ID of the parent run
         **kwargs: Any # Additional arguments
     ) -> Any
     ```

4. `on_agent_finish`: Runs when the agent finishes its execution.

   * Receives the final result returned by the agent.

   - **Syntax:**

     ```python
     on_agent_finish(
         self,
         finish: AgentFinish, # Final result of the agent
         *,
         run_id: UUID, # ID of the current agent run
         parent_run_id: UUID | None = None, # ID of the parent run
         **kwargs: Any # Additional arguments
     ) -> Any
     ```

`on_chain_end` and `on_chain_error` are used for chains, while `on_agent_action` and `on_agent_finish` are used for agents.


In [ ]:
from langchain_openai import ChatOpenAI  # AI model
from langchain_core.callbacks import BaseCallbackHandler  # Callback base class
from langchain_core.tools import Tool  # Creates a tool
from langchain_classic.agents import initialize_agent, AgentType  # Creates AI agent

class MyHandler(BaseCallbackHandler):  # Custom callback handler
    def on_agent_action(self, action, **kwargs):  # Runs when AI selects a tool
        print("Agent action:", action.tool)

    def on_agent_finish(self, finish, **kwargs):  # Runs when AI gives final answer
        print("Agent finished:", finish.return_values["output"])

    def on_chain_end(self, outputs, **kwargs):  # Runs when agent chain completes
        print("Chain completed")

    def on_chain_error(self, error, **kwargs):  # Runs when agent chain fails
        print("Chain error:", error)

def calculator(expression):  # Calculator function
    return str(eval(expression))  # Calculates the expression

tool = Tool(name="Calculator", func=calculator, description="Calculates maths expressions")  # AI tool
model = ChatOpenAI(model="gpt-4.1-mini", temperature=0)  # Creates AI model

agent = initialize_agent(
    tools=[tool],  # Gives calculator to AI
    llm=model,  # Connects AI model
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,  # Lets AI choose the tool
    callbacks=[MyHandler()]  # Connects callbacks
)

agent.invoke("Use the calculator to find 10 + 20")  # Runs the AI agent

'''
Agent action: Calculator
Agent finished: 30
Chain completed
'''

## Inherited from `ToolManagerMixin`

Mixin used to handle callbacks related to tool execution.
It is useful for monitoring successful tool results and handling tool errors.

### Methods

1. `on_tool_end`: Runs when a tool finishes successfully.

   * Receives the output returned by the tool.

   - **Syntax:**

     ```python
     on_tool_end(
         self,
         output: Any, # Output returned by the tool
         *,
         run_id: UUID, # ID of the current tool run
         parent_run_id: UUID | None = None, # ID of the parent run
         **kwargs: Any # Additional arguments
     ) -> Any
     ```

2. `on_tool_error`: Runs when an error occurs during tool execution.

   * Helps in logging, debugging, and handling tool failures.

   - **Syntax:**

     ```python
     on_tool_error(
         self,
         error: BaseException, # Error that occurred
         *,
         run_id: UUID, # ID of the current tool run
         parent_run_id: UUID | None = None, # ID of the parent run
         **kwargs: Any # Additional arguments
     ) -> Any
     ```

`on_tool_end` is triggered when the tool returns a result successfully, while `on_tool_error` is triggered when the tool fails.


In [ ]:
from langchain_core.tools import tool  # Creates a tool
from langchain_core.callbacks import BaseCallbackHandler  # Callback base class

class MyHandler(BaseCallbackHandler):  # Handles tool events
    def on_tool_end(self, output, **kwargs):  # Runs when tool succeeds
        print("Tool result:", output)

    def on_tool_error(self, error, **kwargs):  # Runs when tool fails
        print("Tool error:", error)

@tool
def divide(a: int, b: int) -> float:  # Simple calculator tool
    """Divide two numbers."""
    return a / b  # Returns result or raises division error

handler = MyHandler()  # Creates callback handler

divide.invoke({"a": 10, "b": 2}, config={"callbacks": [handler]})  # Success

try:
    divide.invoke({"a": 10, "b": 0}, config={"callbacks": [handler]})  # Error
except ZeroDivisionError:
    pass  # Prevents program from stopping

## Inherited from `CallbackManagerMixin`

Mixin used to handle callbacks when an LLM, chat model, retriever, chain, or tool **starts running**.
It is useful for logging inputs, monitoring execution, and understanding which AI component has started.

### Methods

1. `on_llm_start`: Runs when a non-chat text-completion LLM starts.

   * Use it for regular LLMs that receive text prompts.
   * For chat models, use `on_chat_model_start`.

   - **Syntax:**

     ```python
     on_llm_start(
         self,
         serialized: dict[str, Any], # Information about the LLM
         prompts: list[str], # Prompts given to the LLM
         *,
         run_id: UUID, # ID of the current LLM run
         parent_run_id: UUID | None = None, # ID of the parent run
         tags: list[str] | None = None, # Tags attached to the run
         metadata: dict[str, Any] | None = None, # Extra run information
         **kwargs: Any # Additional arguments
     ) -> Any
     ```

2. `on_chat_model_start`: Runs when a chat model starts.

   * Receives messages such as system, human, and AI messages.
   * `serialized` and `messages` must be explicitly included in the method signature.

   - **Syntax:**

     ```python
     on_chat_model_start(
         self,
         serialized: dict[str, Any], # Information about the chat model
         messages: list[list[BaseMessage]], # Messages given to the model
         *,
         run_id: UUID, # ID of the current model run
         parent_run_id: UUID | None = None, # ID of the parent run
         tags: list[str] | None = None, # Tags attached to the run
         metadata: dict[str, Any] | None = None, # Extra run information
         **kwargs: Any # Additional arguments
     ) -> Any
     ```

3. `on_retriever_start`: Runs when a retriever starts searching for documents.

   * Receives the user's search query.

   - **Syntax:**

     ```python
     on_retriever_start(
         self,
         serialized: dict[str, Any], # Information about the retriever
         query: str, # Query searched by the retriever
         *,
         run_id: UUID, # ID of the current retrieval run
         parent_run_id: UUID | None = None, # ID of the parent run
         tags: list[str] | None = None, # Tags attached to the run
         metadata: dict[str, Any] | None = None, # Extra run information
         **kwargs: Any # Additional arguments
     ) -> Any
     ```

4. `on_chain_start`: Runs when a chain starts.

   * Receives the input passed to the chain.

   - **Syntax:**

     ```python
     on_chain_start(
         self,
         serialized: dict[str, Any], # Information about the chain
         inputs: dict[str, Any], # Inputs given to the chain
         *,
         run_id: UUID, # ID of the current chain run
         parent_run_id: UUID | None = None, # ID of the parent run
         tags: list[str] | None = None, # Tags attached to the run
         metadata: dict[str, Any] | None = None, # Extra run information
         **kwargs: Any # Additional arguments
     ) -> Any
     ```

5. `on_tool_start`: Runs when a tool starts executing.

   * Receives the input passed to the tool.

   - **Syntax:**

     ```python
     on_tool_start(
         self,
         serialized: dict[str, Any], # Information about the tool
         input_str: str, # Tool input in string form
         *,
         run_id: UUID, # ID of the current tool run
         parent_run_id: UUID | None = None, # ID of the parent run
         tags: list[str] | None = None, # Tags attached to the run
         metadata: dict[str, Any] | None = None, # Extra run information
         inputs: dict[str, Any] | None = None, # Tool input as a dictionary
         **kwargs: Any # Additional arguments
     ) -> Any
     ```

`CallbackManagerMixin` mainly handles **start events**, while other mixins handle successful completion and errors.


In [ ]:
from langchain_core.callbacks import BaseCallbackHandler  # Callback base class
from langchain_core.language_models import FakeListLLM, FakeListChatModel  # Fake AI models
from langchain_core.retrievers import BaseRetriever  # Retriever base class
from langchain_core.documents import Document  # Stores documents
from langchain_core.runnables import RunnableLambda  # Creates a simple chain
from langchain_core.tools import tool  # Creates a tool

class MyHandler(BaseCallbackHandler):  # Handles start events
    def on_llm_start(self, serialized, prompts, **kwargs):  # Non-chat LLM starts
        print("LLM started:", prompts[0])

    def on_chat_model_start(self, serialized, messages, **kwargs):  # Chat model starts
        print("Chat model started:", messages[0][0].content)

    def on_retriever_start(self, serialized, query, **kwargs):  # Retriever starts
        print("Retriever started:", query)

    def on_chain_start(self, serialized, inputs, **kwargs):  # Chain starts
        print("Chain started:", inputs)

    def on_tool_start(self, serialized, input_str, **kwargs):  # Tool starts
        print("Tool started:", input_str)

class MyRetriever(BaseRetriever):  # Simple retriever
    def _get_relevant_documents(self, query):  # Searches documents
        return [Document(page_content="Python is easy")]

@tool
def add(a: int, b: int) -> int:  # Simple tool
    """Add two numbers."""
    return a + b

handler = MyHandler()  # Creates callback handler

FakeListLLM(responses=["Hello"]).invoke("Hi", config={"callbacks": [handler]})  # LLM
FakeListChatModel(responses=["Hello"]).invoke("Hi", config={"callbacks": [handler]})  # Chat model
MyRetriever().invoke("Python", config={"callbacks": [handler]})  # Retriever
RunnableLambda(lambda x: x * 2).invoke(5, config={"callbacks": [handler]})  # Chain
add.invoke({"a": 2, "b": 3}, config={"callbacks": [handler]})  # Tool

'''
LLM started: Hi
Chat model started: Hi
Retriever started: Python
Chain started: 5
Tool started: {'a': 2, 'b': 3}
'''

## Inherited from `RunManagerMixin`

Mixin used to handle general run events.
It is useful for displaying intermediate text, monitoring retry attempts, and handling user-defined custom events.

### Methods

1. `on_text`: Runs when arbitrary or intermediate text is generated during execution.

   * Useful for displaying progress messages or intermediate results.

   - **Syntax:**

     ```python
     on_text(
         self,
         text: str, # Text generated during execution
         *,
         run_id: UUID, # ID of the current run
         parent_run_id: UUID | None = None, # ID of the parent run
         **kwargs: Any # Additional arguments
     ) -> Any
     ```

2. `on_retry`: Runs whenever an operation is retried.

   * Useful for monitoring failed attempts and retry behaviour.

   - **Syntax:**

     ```python
     on_retry(
         self,
         retry_state: RetryCallState, # Information about the retry attempt
         *,
         run_id: UUID, # ID of the current run
         parent_run_id: UUID | None = None, # ID of the parent run
         **kwargs: Any # Additional arguments
     ) -> Any
     ```

3. `on_custom_event`: Runs when a user-defined custom event is triggered.

   * Useful for handling application-specific events such as progress updates or status changes.

   - **Syntax:**

     ```python
     on_custom_event(
         self,
         name: str, # Name of the custom event
         data: Any, # Data sent with the event
         *,
         run_id: UUID, # ID of the current run
         tags: list[str] | None = None, # Tags attached to the event
         metadata: dict[str, Any] | None = None, # Extra event information
         **kwargs: Any # Additional arguments
     ) -> Any
     ```

`on_text` handles intermediate text, `on_retry` handles retry attempts, and `on_custom_event` handles application-specific events.


In [1]:
from tenacity import Retrying, stop_after_attempt  # Retry support
from langchain_core.callbacks import BaseCallbackHandler  # Callback base class
from langchain_core.callbacks.manager import dispatch_custom_event  # Custom event
from langchain_core.language_models import FakeListLLM  # Fake AI model
from langchain_core.runnables import RunnableLambda  # Creates runnable workflow

class MyHandler(BaseCallbackHandler):  # Receives events
    def on_text(self, text, **kwargs):  # Receives intermediate text
        print("Text:", text)

    def on_retry(self, retry_state, **kwargs):  # Runs before retry
        print("Retrying AI...")

    def on_custom_event(self, name, data, **kwargs):  # Receives custom event
        print(name, ":", data)

class AIWorkflow:  # Production-style AI workflow
    def __init__(self, llm):
        self.llm = llm  # Stores the AI model
        self.first_attempt = True  # Simulates one temporary failure

    def __call__(self, question, run_manager, config):
        run_manager.on_text(f"Processing: {question}")  # Sends text event
        dispatch_custom_event("Progress", "Calling AI model", config=config)  # Sends custom event

        for attempt in Retrying(stop=stop_after_attempt(2), before_sleep=run_manager.on_retry):
            with attempt:
                if self.first_attempt:  # Simulates an API timeout
                    self.first_attempt = False
                    raise TimeoutError("Temporary AI error")
                return self.llm.invoke(question, config=config)  # Calls fake AI

llm = FakeListLLM(responses=["AI means Artificial Intelligence."])  # Fake response
app = RunnableLambda(AIWorkflow(llm))  # Creates AI application

answer = app.invoke("What is AI?", config={"callbacks": [MyHandler()]})  # Runs application
print("Answer:", answer)  # Displays AI response

Text: Processing: What is AI?
Progress : Calling AI model
Retrying AI...
Answer: AI means Artificial Intelligence.
